# Classificador MNIST com uma MLP simples (PyTorch) — versão revisada

Este notebook parte do seu `first-nn-pytorch1.ipynb` (uma MLP simples — 784 → 128 → 64 → 10 — treinada no MNIST) e mantém a mesma ideia central: uma rede densa simples, sem convoluções, como primeiro contato com PyTorch. Nada na arquitetura ou no problema mudou — o que foi ajustado foi **como o treinamento é acompanhado e como o resultado é avaliado**.

## Diagnóstico do notebook original

Não encontrei nenhum bug crítico (a combinação `log_softmax` + `NLLLoss` está correta, o split treino/teste é o particionamento canônico do MNIST — não há vazamento de dados, e o laço de treino atualiza os pesos corretamente). Os pontos que valem correção são de **rigor de avaliação** e **boas práticas de engenharia**:

**Importante**
1. O loss impresso a cada época é o do **último mini-batch**, não a média da época — é uma leitura enganosa do quão bem o treino foi naquela época (um único batch "sortudo" ou "azarado" no final domina o número exibido).
2. Não há **nenhum acompanhamento** de treino/teste ao longo das épocas (sem histórico salvo, sem curva). Sem isso, não dá para saber se o modelo está com overfitting, underfitting, ou se 5 épocas foi pouco ou demais.
3. A avaliação final é só a **acurácia global no teste**. Não há baseline de comparação, nem métricas por classe (precision/recall/F1), nem matriz de confusão — então não se sabe quais dígitos o modelo confunde mais (classicamente, 4↔9 e 3↔5↔8).
4. Sem **seed fixa** — cada execução do notebook pode chegar a um modelo (e a uma acurácia final) ligeiramente diferente, dificultando reproduzir o resultado.

**Melhoria**
5. Sem gerenciamento de **device** (`cuda`/`cpu`) — roda sempre na CPU mesmo que haja GPU disponível.
6. Sem alternância explícita `model.train()` / `model.eval()` — não muda o resultado nesta arquitetura específica (não há `Dropout`/`BatchNorm`), mas é um hábito que vale criar desde já, porque uma futura versão com `Dropout` se comportaria errado silenciosamente sem isso.
7. Normalização com `(0.5, 0.5)` em vez da média/desvio-padrão real do MNIST (`0.1307`/`0.3081`) — funciona bem na prática (é uma aproximação comum em tutoriais), mas vale registrar a diferença.
8. Poucos comentários explicando as decisões — dado que este é um notebook de aprendizado/portfólio.

## O que foi adicionado

Loss e acurácia média por época (treino **e** teste, não só o `.item()` do último batch), curvas de treino vs. teste, um baseline trivial (classificador de "classe majoritária") para contextualizar a acurácia da rede, métricas por classe (`classification_report`) e matriz de confusão, seeds fixas, e gerenciamento de device. A arquitetura da rede (MLP 784→128→64→10) e a lógica central de treino foram mantidas.


## Etapa 1 — Imports, reprodutibilidade e device

Além dos imports originais, fixamos as sementes de aleatoriedade (para que a inicialização dos pesos, o shuffle do `DataLoader` e o resultado final sejam reprodutíveis) e detectamos automaticamente se há GPU disponível.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Sementes fixas: sem isso, a inicializacao dos pesos da rede e a ordem de
# shuffle do DataLoader mudam a cada execucao, e o resultado final (mesmo
# rodando o notebook duas vezes sem alterar nada) pode variar. Isso dificulta
# comparar experimentos e reproduzir o que esta reportado aqui.
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Usa GPU se disponivel; senao, cai para CPU. Isso nao muda o resultado,
# so a velocidade do treino -- mas precisa que o modelo E os dados estejam
# no mesmo device, por isso e definido aqui, no topo do notebook.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Etapa 2 — Dados: carregamento e transformação

`ToTensor()` converte as imagens (originalmente valores inteiros 0–255) para tensores float em [0, 1]. `Normalize((0.5,), (0.5,))` reescala isso para aproximadamente [-1, 1] — uma aproximação simples e comum; o valor mais preciso, calculado a partir do próprio dataset, seria `Normalize((0.1307,), (0.3081,))` (média e desvio-padrão reais dos pixels do MNIST). Mantive a versão original por já funcionar bem e não ser o ponto que mais importa aqui — mas vale saber que essa alternativa existe.

In [ ]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.5,), (0.5,)),
])

# O MNIST ja vem com uma separacao canonica treino/teste (60.000 / 10.000
# imagens) definida pelos proprios criadores do dataset -- nao estamos nos
# fazendo esse split, entao nao ha risco de vazamento de dados aqui.
train_set = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=32, shuffle=False)

## Etapa 3 — Inspeção rápida dos dados

In [ ]:
print("Numero de imagens no conjunto de treino:", len(train_set))
print("Numero de imagens no conjunto de teste:", len(test_set))
print(f"Formato de cada imagem: {train_loader.dataset[0][0].shape}")

## Etapa 4 — Visualização de amostras

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 3))
for i in range(10):
    axes[i].imshow(train_loader.dataset[i][0].squeeze(), cmap="gray")
    axes[i].set_title(train_loader.dataset[i][1])
    axes[i].axis("off")
plt.show()

## Etapa 5 — Definição do modelo (MLP simples)

Mesma arquitetura do notebook original: achata a imagem 28×28 num vetor de 784 valores, passa por duas camadas ocultas (128 e 64 neurônios, com ReLU) e termina numa camada de 10 saídas com `log_softmax` — combinado com `NLLLoss` no treino (Etapa 7), isso é matematicamente equivalente a usar logits crus com `CrossEntropyLoss`. `.to(device)` move os parâmetros do modelo para a GPU (se houver) antes do treino.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.input = nn.Linear(28 * 28, 128)
        self.hidden = nn.Linear(128, 64)
        self.output = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = F.relu(self.input(x))
        x = F.relu(self.hidden(x))
        x = F.log_softmax(self.output(x), dim=1)
        return x


model = NeuralNetwork().to(device)
total_parametros = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"Total de parametros treinaveis: {total_parametros:,}")

## Etapa 6 — Baseline de comparação (classificador trivial)

Antes de treinar qualquer rede, vale saber o piso de comparação: um classificador "burro" que sempre prevê a classe mais frequente no treino. Com 10 classes razoavelmente balanceadas, esse baseline deve ficar perto de 10% de acurácia — qualquer coisa muito próxima disso, mesmo depois de treinar uma rede neural, seria sinal de que algo no pipeline está errado.

In [ ]:
# Conta a frequencia de cada digito no conjunto de treino e usa a classe
# mais frequente como previsao fixa para todo o conjunto de teste.
labels_treino = train_set.targets.numpy()
classe_majoritaria = np.bincount(labels_treino).argmax()

labels_teste = test_set.targets.numpy()
previsoes_baseline = np.full_like(labels_teste, fill_value=classe_majoritaria)
acuracia_baseline = (previsoes_baseline == labels_teste).mean()

print(f"Classe majoritaria no treino: digito {classe_majoritaria}")
print(f"Acuracia do baseline (sempre prever '{classe_majoritaria}') no teste: {acuracia_baseline * 100:.2f}%")

## Etapa 7 — Treinamento com acompanhamento de treino e teste por época

Diferente do notebook original (que só imprimia o loss do último mini-batch), aqui cada época calcula a **média** do loss e da acurácia sobre TODO o conjunto de treino, e também avalia no conjunto de teste ao final da época (com `model.eval()` + `torch.no_grad()`, sem atualizar pesos). Isso é o que permite montar as curvas da próxima etapa e diagnosticar overfitting.

In [ ]:
loss_function = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 5

historico = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

for epoch in range(epochs):
    # --- fase de treino ---
    model.train()
    perda_acumulada, acertos, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        output = model(images)
        loss = loss_function(output, labels)
        loss.backward()
        optimizer.step()

        perda_acumulada += loss.item() * images.size(0)
        acertos += (output.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    train_loss = perda_acumulada / total
    train_acc = acertos / total

    # --- fase de avaliacao no teste (a cada epoca, so para acompanhar) ---
    model.eval()
    perda_acumulada, acertos, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            output = model(images)
            loss = loss_function(output, labels)
            perda_acumulada += loss.item() * images.size(0)
            acertos += (output.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    test_loss = perda_acumulada / total
    test_acc = acertos / total

    historico["train_loss"].append(train_loss)
    historico["train_acc"].append(train_acc)
    historico["test_loss"].append(test_loss)
    historico["test_acc"].append(test_acc)

    print(f"Epoca [{epoch + 1}/{epochs}] "
          f"train_loss: {train_loss:.4f}  train_acc: {train_acc * 100:.2f}%  "
          f"test_loss: {test_loss:.4f}  test_acc: {test_acc * 100:.2f}%")

## Etapa 8 — Curvas de loss e acurácia (treino vs. teste)

**Como interpretar:** se a curva de treino continua melhorando enquanto a de teste estagna ou piora, é overfitting. Se as duas melhoram juntas e convergem, o modelo está generalizando bem — o que é esperado no MNIST com essa arquitetura, dado que é um problema relativamente fácil para o tamanho da rede.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(historico["train_loss"], label="Treino", marker="o")
ax1.plot(historico["test_loss"], label="Teste", marker="o")
ax1.set_title("Loss por epoca")
ax1.set_xlabel("Epoca")
ax1.set_ylabel("NLLLoss")
ax1.legend()

ax2.plot([a * 100 for a in historico["train_acc"]], label="Treino", marker="o")
ax2.plot([a * 100 for a in historico["test_acc"]], label="Teste", marker="o")
ax2.set_title("Acuracia por epoca")
ax2.set_xlabel("Epoca")
ax2.set_ylabel("Acuracia (%)")
ax2.legend()

plt.tight_layout()
plt.show()

## Etapa 9 — Avaliação final: métricas por classe e matriz de confusão

Acurácia sozinha esconde *onde* o modelo erra. O `classification_report` traz precision/recall/F1 por dígito (e a média macro), e a matriz de confusão mostra exatamente quais pares de dígitos são mais confundidos entre si.

In [ ]:
model.eval()
todas_previsoes, todos_rotulos = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        output = model(images)
        previsoes = output.argmax(dim=1).cpu().numpy()
        todas_previsoes.extend(previsoes)
        todos_rotulos.extend(labels.numpy())

todas_previsoes = np.array(todas_previsoes)
todos_rotulos = np.array(todos_rotulos)
acuracia_final = (todas_previsoes == todos_rotulos).mean()

print("=" * 70)
print("COMPARACAO: REDE NEURAL vs. BASELINE (classe majoritaria)")
print("=" * 70)
print(f"  Baseline -> acuracia: {acuracia_baseline * 100:.2f}%")
print(f"  Rede MLP -> acuracia: {acuracia_final * 100:.2f}%")
print("=" * 70)

print("\nRelatorio de classificacao (precision / recall / F1 por digito):\n")
print(classification_report(todos_rotulos, todas_previsoes, digits=4))

# Matriz de confusao: cada celula (i, j) e quantas imagens do digito i
# foram classificadas pelo modelo como digito j. A diagonal principal e o
# que o modelo acertou; fora dela sao os erros -- util para ver, por
# exemplo, se 4 e 9 estao sendo confundidos entre si.
cm = confusion_matrix(todos_rotulos, todas_previsoes)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("Previsto")
ax.set_ylabel("Real")
ax.set_title("Matriz de Confusao - Conjunto de Teste")
for i in range(10):
    for j in range(10):
        cor_texto = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, cm[i, j], ha="center", va="center", color=cor_texto, fontsize=8)
plt.colorbar(im)
plt.tight_layout()
plt.show()

## Etapa 10 — Inspeção qualitativa (uma imagem)

Mesma verificação do notebook original: olhar uma previsão individual, com as probabilidades por classe.

In [ ]:
def view_classify(image, probabilities):
    probabilities = probabilities.data.cpu().numpy().squeeze()

    fig, (ax1, ax2) = plt.subplots(figsize=(6, 9), ncols=2)
    ax1.imshow(image.cpu().numpy().squeeze())
    ax1.axis("off")
    ax2.barh(np.arange(10), probabilities)
    ax2.set_aspect(0.1)
    ax2.set_yticks(np.arange(10))
    ax2.set_yticklabels(np.arange(10))
    ax2.set_title("Class Probability")
    ax2.set_xlim(0, 1.1)
    plt.tight_layout()
    plt.show()


images, _ = next(iter(test_loader))
image = images[0].to(device)

model.eval()
with torch.no_grad():
    log_probabilities = model(image.unsqueeze(0))

probabilities = torch.exp(log_probabilities)
view_classify(image.view(1, 28, 28), probabilities)

## Conclusões e limitações

A MLP treinada supera claramente o baseline de classe majoritária (a comparação numérica está na Etapa 9) — nesse caso, isso é o esperado: o MNIST é um dos benchmarks mais fáceis e bem-comportados de visão computacional, então uma rede densa simples já costuma passar de 95% de acurácia com poucas épocas.

**Limitações a ter em mente:**
- O MNIST é "fácil" precisamente por ser limpo, balanceado e de baixa resolução — um bom resultado aqui não se transfere automaticamente para problemas de classificação de imagem do mundo real (fundos variados, iluminação, classes desbalanceadas).
- Uma extensão natural (fora do escopo desta revisão, que preservou a arquitetura original) seria trocar a MLP por uma CNN (`nn.Conv2d`), que costuma superar MLPs em imagens por explorar a estrutura espacial dos pixels — vale como próximo passo de estudo, não como correção obrigatória deste notebook.
- Os hiperparâmetros (5 épocas, `lr=0.001`, batch 32) não foram otimizados via busca sistemática; funcionam bem o suficiente para este problema fácil, mas não foram validados via nenhuma forma de tuning.
